# 01 - Fundamentos del Clasificador Naïve Bayes y Variantes

## Curso de Machine Learning: Modelos de Clasificación

El clasificador **Naïve Bayes (Bayes Ingenuo)** es uno de los algoritmos probabilísticos más veloces, elegantes y computacionalmente escalables del aprendizaje automático. Fundamentado en el célebre **Teorema de Bayes** formulado en el siglo XVIII por Thomas Bayes, este modelo calcula la probabilidad condicional a posteriori de pertenencia a una clase.

---

### Contenidos
1. Formulación Probabilística del Teorema de Bayes en Clasificación
2. La Asunción "Ingenua" de Independencia Condicional
3. Regla de Decisión MAP (Maximum A Posteriori)
4. Variantes Principales en Scikit-Learn: `GaussianNB`, `MultinomialNB`, `BernoulliNB`
5. Suavizado de Laplace (Laplace Smoothing)
6. Ajuste Práctico de `GaussianNB` e Inspección de Parámetros ($\mu, \sigma^2$)
7. Conclusiones


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.datasets import load_breast_cancer, load_iris
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)
print("Bibliotecas y entorno listos.")


## 1. El Teorema de Bayes en Clasificación

Dado un vector de características $\mathbf{x} = (x_1, x_2, \dots, x_D)^T$, deseamos predecir la probabilidad condicional a posteriori de que pertenezca a la clase $c \in \{1, \dots, C\}$:

$$P(Y = c \mid \mathbf{X} = \mathbf{x}) = \frac{P(Y = c) \cdot P(\mathbf{X} = \mathbf{x} \mid Y = c)}{P(\mathbf{X} = \mathbf{x})}$$

Donde:
- **Prior $P(Y = c)$**: Probabilidad previa de observar la clase $c$ (frecuencia relativa en los datos).
- **Verosimilitud (Likelihood) $P(\mathbf{x} \mid Y = c)$**: Probabilidad de observar el patrón de características $\mathbf{x}$ dado que la muestra proviene de la clase $c$.
- **Evidencia $P(\mathbf{x})$**: Constante de normalización idéntica para todas las clases ($P(\mathbf{x}) = \sum_k P(Y=k)P(\mathbf{x}\mid Y=k)$).
- **Posterior $P(Y = c \mid \mathbf{x})$**: Probabilidad actualizada tras observar los datos.


## 2. La Asunción "Ingenua" de Independencia Condicional

Calcular la función de distribución conjunta multivariada $P(x_1, \dots, x_D \mid Y=c)$ requiere estimar una cantidad exponencial de parámetros.

**La simplificación de Naïve Bayes**: Asume que todas las características $x_j$ son **mutuamente independientes** dadas la clase $Y=c$:

$$P(\mathbf{x} \mid Y = c) = \prod_{j=1}^D P(x_j \mid Y = c)$$

### Regla de Decisión MAP (Maximum A Posteriori):
Para predecir, maximizamos el producto (o la suma en escala logarítmica para estabilidad numérica):

$$\hat{y} = \arg\max_{c} \left[ \ln P(Y = c) + \sum_{j=1}^D \ln P(x_j \mid Y = c) \right]$$


## 3. Variantes del Clasificador en Scikit-Learn

Dependiendo de la naturaleza estadística de las variables, Scikit-Learn ofrece distintas implementaciones:

| Variante | Tipo de Variables | Distribución Asumida | Caso de Uso Típico |
|---|---|---|---|
| `GaussianNB` | Cuantitativas continuas | Normal Gaussiana: $\mathcal{N}(\mu_{cj}, \sigma^2_{cj})$ | Datos tabulares clínicos y físicos |
| `MultinomialNB` | Conteos discretos | Multinomial / Frecuencias | Clasificación de texto (Bag of Words) |
| `BernoulliNB` | Binarias / Booleanas | Bernoulli ($\{0, 1\}$) | Detección de presencia/ausencia de términos |
| `ComplementNB` | Conteos discretos | Multinomial corregida | Datasets de texto desbalanceados |


In [ ]:
# Ajuste de GaussianNB en Dataset Cancer de Mama
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

gnb = GaussianNB()
gnb.fit(X_train, y_train)

y_pred_gnb = gnb.predict(X_test)
acc_gnb = accuracy_score(y_test, y_pred_gnb)

print(f"Exactitud de GaussianNB en Test: {acc_gnb*100:.2f}%")
print(f"Probabilidades previas estimadas (class_prior_): {np.round(gnb.class_prior_, 3)}")


In [ ]:
# Inspeccion de las distribuciones gaussianas aprendidas (theta_ = medias, var_ = varianzas)
feature_idx = 0 # 'mean radius'
feature_name = cancer.feature_names[feature_idx]

mu_maligno, var_maligno = gnb.theta_[0, feature_idx], gnb.var_[0, feature_idx]
mu_benigno, var_benigno = gnb.theta_[1, feature_idx], gnb.var_[1, feature_idx]

x_axis = np.linspace(5, 30, 500)
pdf_maligno = stats.norm.pdf(x_axis, mu_maligno, np.sqrt(var_maligno))
pdf_benigno = stats.norm.pdf(x_axis, mu_benigno, np.sqrt(var_benigno))

plt.figure(figsize=(10, 4.8))
plt.plot(x_axis, pdf_maligno, color='#d62728', lw=2.5, label=f'Maligno: N(mu={mu_maligno:.1f}, sigma={np.sqrt(var_maligno):.1f})')
plt.plot(x_axis, pdf_benigno, color='#2ca02c', lw=2.5, label=f'Benigno: N(mu={mu_benigno:.1f}, sigma={np.sqrt(var_benigno):.1f})')
plt.axvline((mu_maligno + mu_benigno)/2, color='gray', linestyle=':', label='Zona de Cruce de Probabilidad')
plt.xlabel(feature_name, fontweight='bold')
plt.ylabel('Densidad de Probabilidad (Likelihood)', fontweight='bold')
plt.title(f'Distribuciones Gaussianas Estimadas para la Caracteristica: {feature_name}', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()


## 4. Suavizado de Laplace (Laplace Smoothing)

En variables discretas o procesamiento de texto (`MultinomialNB`), si una palabra nunca aparece en los documentos de la clase $c$ durante el entrenamiento:
$$P(x_j \mid Y=c) = 0 \implies \prod_{j=1}^D P(x_j \mid Y=c) = 0$$

Una sola palabra ausente anularía la probabilidad de todas las demás palabras del documento.
Para resolver este fallo, aplicamos el **Suavizado de Laplace (parámetro `alpha > 0`)**:

$$\hat{P}(x_j \mid Y=c) = \frac{N_{cj} + \alpha}{N_c + \alpha D}$$

Al fijar $\alpha = 1$, agregamos una pseudo-cuenta unitaria a cada palabra posible, asegurando que ninguna probabilidad sea estrictamente cero.


## 5. Conclusiones

1. **Eficiencia Extrema**: El entrenamiento de Naïve Bayes requiere solo una única pasada por los datos para computar medias y varianzas ($O(N \cdot D)$).
2. **Escala y Big Data**: Es ideal para escenarios con streaming masivo de datos o memoria limitada.
3. **Robustez ante Características Irrelevantes**: Las variables no informativas tienen distribuciones similares entre clases, aportando un factor multiplicativo neutro.

En el siguiente cuaderno, **02_Aplicacion_Gaussian_y_Clasificacion_Texto.ipynb**, aplicaremos `MultinomialNB` a procesamiento de lenguaje natural (NLP) y realizaremos un benchmark comparativo de velocidad frente a otros clasificadores.
